# Lack of Adaptive Capacity leave-one-out scenarios

This notebook loads all four raster layers of the lack of adaptive capacity variables and aggregates them through simple additive aggregation. However, it does so within a loop and leaves one of the variables out in every iteration to generate different scenarios. It than creates: 
- a raster with mean lack of adaptive capacity scores at a chosen pixel size for four different scenarios
- a raster with lack of adaptive capacity score at a chosen pixel size (not normalized) for four different scenarios
- a raster with missing variable counts at a chosen pixel size for four different scenarios

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `conflict_5000m.tif`
- `landrights_5000m.tif`
- `rule_of_law_5000m.tif`
- `edi_5000m.tif`
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)


In [ ]:
# Configuration (edit these paths if needed)
BII_5000M =  'sensitivity\\biodiversity_intactness\\bii_5000m.tif'  
WORLD_COUNTRIES_GENERAL = 'lackof_adapt\\World_Countries_(Generalized)_8414823838130214587.gpkg' 
CONFLICT_5000M = 'lackof_adapt\\conflict\\conflict_5000m.tif'
EDI_5000M = 'lackof_adapt\\environmental_democracy\\edi_5000m.tif'
LANDRIGHTS_5000M = 'lackof_adapt\\landrights\\landrights_5000m.tif'  
RULE_OF_LAW_5000M = 'lackof_adapt\\rule_of_law\\rule_of_law_5000m.tif'
LACKOF_ADAPT = 'lackof_adapt\\lackof_adapt.tif'
MISSING_COUNT_LACKOF_ADA = 'lackof_adapt\\missing_count_lackof_adapt.tif'
COUNTRY_MASK = 'lackof_adapt\\scenarios_country_mask.tif'
LACKOF_ADAPT_MEAN = 'lackof_adapt\\lackof_adapt_mean.tif'  

In [ ]:
#import packages
import math
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio import warp
from rasterio import features
import geopandas as gpd
from shapely.geometry import box
from rasterio.features import geometry_mask
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from shapely.ops import unary_union
from rasterio.features import rasterize
import matplotlib.ticker as mticker
from matplotlib.colors import ListedColormap, BoundaryNorm


In [ ]:
#summing up lack of adaptive capacity variables and creating layers to show missing variables and completeness

#Inputs
ref_path = BII_5000M
countries_path = WORLD_COUNTRIES_GENERAL

aligned_rasters = [
     CONFLICT_5000M,
    EDI_5000M,
    LANDRIGHTS_5000M,
    RULE_OF_LAW_5000M
]

window_size = 2048
dst_nodata = -9999.0

out_sum = LACKOF_ADAPT
out_missing = MISSING_COUNT_LACKOF_ADA
out_mask = COUNTRY_MASK
out_mean = LACKOF_ADAPT_MEAN


#load reference grid
with rasterio.open(ref_path) as ref:
    ref_meta = ref.meta.copy()
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height


# Windowed country rasterization function
def build_country_mask():
    gdf = gpd.read_file(countries_path).to_crs(ref_crs)

    # Keep only valid geometries and drop Antarctica
    gdf = gdf[gdf["COUNTRY"] != "Antarctica"].copy()
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[gdf.is_valid].copy()
    gdf = gdf[gdf.geometry.notnull()].copy()

    # spatial index for fast window intersection
    sindex = gdf.sindex

    #prepare output raster metadata for country mask
    mask_meta = ref_meta.copy()
    mask_meta.update(
        dtype="uint8",
        count=1,
        nodata=0,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    #start rasterization of country mask
    with rasterio.open(out_mask, "w", **mask_meta) as dst:
        n_rows = math.ceil(ref_height / window_size)
        n_cols = math.ceil(ref_width / window_size)

        for row in range(n_rows):
            for col in range(n_cols):
                x_off = col * window_size
                y_off = row * window_size
                w = min(window_size, ref_width - x_off)
                h = min(window_size, ref_height - y_off)
                window = Window(x_off, y_off, w, h)

                # window bounds in ref CRS
                win_bounds = rasterio.windows.bounds(window, ref_transform)
                win_geom = box(*win_bounds)

                px = max(abs(ref_transform.a), abs(ref_transform.e))
                win_geom = box(*win_bounds).buffer(px)


                # finds polygons that intersect with this window
                cand_idx = list(sindex.intersection(win_geom.bounds))
                if not cand_idx:
                    # no countries in this window: write 0
                    dst.write(np.zeros((h, w), dtype=np.uint8), 1, window=window)
                    continue

                sub = gdf.iloc[cand_idx]
                sub = sub[sub.intersects(win_geom)]
                if sub.empty:
                    dst.write(np.zeros((h, w), dtype=np.uint8), 1, window=window)
                    continue

                # transform for this window
                win_transform = rasterio.windows.transform(window, ref_transform)
                shapes = ((geom, 1) for geom in sub.geometry)
                burned = features.rasterize(
                    shapes=shapes,
                    out_shape=(h, w),
                    transform=win_transform,
                    fill=0,
                    all_touched=False, 
                    dtype="uint8",
                )
                dst.write(burned, 1, window=window)

    print(f"country mask: {out_mask}")



In [ ]:
# Compute different lack of adaptive capacity scenarios by always leaving one variable out

def compute_lackof_adapt_scenario(raster_paths, scenario_tag, out_dir="scenarios"):
    os.makedirs(out_dir, exist_ok=True)

    n_vars = len(raster_paths)

    out_sum = os.path.join(out_dir, f"lackof_adapt_{scenario_tag}.tif")
    out_missing = os.path.join(out_dir, f"missing_count_{scenario_tag}.tif")
    out_mean = os.path.join(out_dir, f"lackof_adapt_mean_{scenario_tag}.tif")

    sum_meta = ref_meta.copy()
    sum_meta.update(
        dtype="float32",
        count=1,
        nodata=dst_nodata,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    count_meta = ref_meta.copy()
    count_meta.update(
        dtype="uint8",
        count=1,
        nodata=255,  # outside/invalid
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )


    mean_meta = ref_meta.copy()
    mean_meta.update(
        dtype="float32",
        count=1,
        nodata=dst_nodata,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256
    )

    srcs = [rasterio.open(p) for p in raster_paths]
    msk = rasterio.open(out_mask)

    try:
        with rasterio.open(out_sum, "w", **sum_meta) as dst_sum, \
             rasterio.open(out_missing, "w", **count_meta) as dst_missing, \
             rasterio.open(out_mean, "w", **mean_meta) as dst_mean:

            n_rows = math.ceil(ref_height / window_size)
            n_cols = math.ceil(ref_width / window_size)

            for row in range(n_rows):
                for col in range(n_cols):
                    x_off = col * window_size
                    y_off = row * window_size
                    w = min(window_size, ref_width - x_off)
                    h = min(window_size, ref_height - y_off)
                    window = Window(x_off, y_off, w, h)

                    country = msk.read(1, window=window).astype(bool)

                    sum_arr = np.zeros((h, w), dtype=np.float32)
                    known = np.zeros((h, w), dtype=np.uint8)

                    for s in srcs:
                        a = s.read(1, window=window)

                        valid = np.isfinite(a) & (a != dst_nodata)
                        use = country & valid
                        if np.any(use):
                            sum_arr[use] += a[use].astype(np.float32)
                            known[use] += 1

                    missing = (n_vars - known).astype(np.uint8)

                    # IMPORTANT: ensure we only write inside countries + where at least one var exists
                    has_data = country & (known > 0)

                    out_sum_arr = np.full((h, w), dst_nodata, dtype=np.float32)
                    out_missing_arr = np.full((h, w), 255, dtype=np.uint8)
                    out_mean_arr = np.full((h, w), dst_nodata, dtype=np.float32)

                    out_sum_arr[has_data] = sum_arr[has_data]
                    out_missing_arr[has_data] = missing[has_data]
                    out_mean_arr[has_data] = sum_arr[has_data] / known[has_data].astype(np.float32)

                    dst_sum.write(out_sum_arr, 1, window=window)
                    dst_missing.write(out_missing_arr, 1, window=window)
                    dst_mean.write(out_mean_arr, 1, window=window)

        print("Wrote scenario outputs:")
        print(" ", out_sum)
        print(" ", out_missing)
        print(" ", out_mean)

    finally:
        for s in srcs:
            s.close()
        msk.close()


In [ ]:
# Run: leave-one-out loop

def run_leave_one_out(aligned_rasters):
    build_country_mask()

    for omit_path in aligned_rasters:
        kept = [p for p in aligned_rasters if p != omit_path]

        # scenario tag based on omitted filename (safe-ish)
        base = os.path.splitext(os.path.basename(omit_path))[0]
        scenario_tag = f"omit_{base}"

        print(f"\nScenario: {scenario_tag}")
        print("Omitting:", omit_path)
        compute_lackof_adapt_scenario(kept, scenario_tag, out_dir="scenarios")


# Execute
run_leave_one_out(aligned_rasters)